In [ ]:
import time
import psutil
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

In [ ]:

# 1. Cargar dataset
# ========================
df = pd.read_csv('data/raw/diabetic_data.csv')

# 2. Preparacion de testeos y settings
# ========================

# Diccionario para almacenar los resultados
results = []

def ejecutar_modelo(nombre, instancia, X_train, y_train, X_test, y_test, opti_info):
    """
    Función general para entrenar y evaluar un modelo con optimización,
    midiendo tiempo y uso de memoria.
    """
    
    # Medir el uso inicial de memoria
    process = psutil.Process()
    memory_before = process.memory_info().rss / (1024 * 1024)  # en MB

    start_time = time.time()

    # Entrenar el modelo
    instancia.fit(X_train, y_train)

    end_time = time.time()

    # Medir el uso de memoria después del entrenamiento
    memory_after = process.memory_info().rss / (1024 * 1024)  # en MB
    memory_used = memory_after - memory_before

    training_time = end_time - start_time
    print(f"Tiempo de entrenamiento: {training_time:.4f} segundos")
    print(f"Uso de memoria durante el entrenamiento: {memory_used:.2f} MB")

    # Hacer predicciones
    y_pred = instancia.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy en el conjunto de prueba: {accuracy:.4f}")

    results.append({
        'Modelo': nombre,
        'Técnica de Optimización': opti_info['technique'],
        'Configuración': opti_info['comments'],
        'Tiempo de Entrenamiento (s)': training_time,
        'Uso de Memoria (MB)': memory_used,
        'Accuracy': accuracy
    })
    return instancia

In [ ]:

# 3. Preparacion de testeos y settings
# ========================

#  K-Nearest Neighbors (KNN) 
# Técnicas de optimización: KD-Trees, Ball Trees
# scikit-learn usa 'kd_tree' o 'ball_tree' como algoritmos para buscar vecinos
knn_optimization_info = {
    'technique': 'KD-Trees (algorithm="kd_tree")',
    'comments': 'Usando KD-Tree para acelerar la búsqueda de vecinos, n_neighbors=5'
}
knn_model = KNeighborsClassifier(n_neighbors=5, algorithm='kd_tree', n_jobs=-1) # n_jobs=-1 para usar todos los cores
ejecutar_modelo("KNN", knn_model, X_train, y_train, X_test, y_test, knn_optimization_info)

knn_optimization_info = {
    'technique': 'Ball Trees (algorithm="ball_tree")',
    'comments': 'Usando Ball Trees para acelerar la búsqueda de vecinos, n_neighbors=5'
}
knn_model = KNeighborsClassifier(n_neighbors=5, algorithm='ball_tree', n_jobs=-1) 
ejecutar_modelo("KNN", knn_model, X_train, y_train, X_test, y_test, knn_optimization_info)



#  Naive Bayes 
# Técnicas de optimización: partial_fit() con entrenamiento por lotes
# Esto es útil para datasets grandes que no caben en memoria.

nb_optimization_info = {
    'technique': 'partial_fit() con entrenamiento por lotes',
    'comments': 'Simulación de entrenamiento por lotes para datos grandes. alpha por defecto para GaussianNB.'
}
gnb_model = GaussianNB()

# Si tus datos son grandes, descomentarías y adaptarías la siguiente lógica:
batch_size = 100
for i in range(0, len(X_train), batch_size):
    X_batch = X_train[i:i + batch_size]
    y_batch = y_train[i:i + batch_size]
    gnb_model.partial_fit(X_batch, y_batch, classes=np.unique(y_train))


#  Ridge 
# Técnicas de optimización: Solver optimizado: saga 
ridge_optimization_info = {
    'technique': 'Solver optimizado: saga',
    'comments': 'Usando RidgeClassifier con penalty="l2" y solver="saga" para eficiencia.'
}
ridge_model = RidgeClassifier(alpha=1.0, solver='saga', random_state=42) 
ejecutar_modelo("Ridge", ridge_model, X_train, y_train, X_test, y_test, ridge_optimization_info)

#  Lasso 
# Técnicas de optimización: Solver optimizado: saga 
lasso_optimization_info = {
    'technique': 'Solver optimizado: saga',
    'comments': 'Usando LogisticRegression con penalty="l1" y solver="saga" para Lasso.'
}

lasso_model = LogisticRegression(penalty='l1', solver='saga', random_state=42, n_jobs=-1)
ejecutar_modelo("Lasso", lasso_model, X_train, y_train, X_test, y_test, lasso_optimization_info)


#  XGBoost 
# Técnicas de optimización: tree_method='hist', early_stopping_rounds
xgb_optimization_info = {
    'technique': "tree_method='hist', early_stopping_rounds",
    'comments': 'tree_method="hist" para un entrenamiento más rápido, early stopping para evitar sobreajuste.'
}
# XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic', # para clasificación binaria
    eval_metric='logloss',      # métrica para early stopping
    use_label_encoder=False,    # suprime la advertencia de depreciación
    tree_method='hist',         # optimización para entrenamiento rápido
    n_estimators=1000,          # un número alto, early stopping lo reducirá
    n_jobs=-1,                  # usar todos los cores
    random_state=42
)

# Para usar early_stopping_rounds, necesitamos un conjunto de validación
X_train_xgb, X_val_xgb, y_train_xgb, y_val_xgb = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

start_time_xgb = time.time()
process_xgb = psutil.Process()
memory_before_xgb = process_xgb.memory_info().rss / (1024 * 1024)

xgb_model.fit(X_train_xgb, y_train_xgb,
              eval_set=[(X_val_xgb, y_val_xgb)],
              early_stopping_rounds=50, # parar si no mejora en 50 rondas
              verbose=False) # para no mostrar cada ronda

end_time_xgb = time.time()
memory_after_xgb = process_xgb.memory_info().rss / (1024 * 1024)
memory_used_xgb = memory_after_xgb - memory_before_xgb

training_time_xgb = end_time_xgb - start_time_xgb
y_pred_xgb = xgb_model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print("\n--- Optimizando XGBoost ---")
print(f"Técnica de Optimización: {xgb_optimization_info['technique']}")
print(f"Comentarios de Configuración: {xgb_optimization_info['comments']}")
print(f"Tiempo de entrenamiento: {training_time_xgb:.4f} segundos")
print(f"Uso de memoria durante el entrenamiento: {memory_used_xgb:.2f} MB")
print(f"Accuracy en el conjunto de prueba: {accuracy_xgb:.4f}")

results.append({
    'Modelo': "XGBoost",
    'Técnica de Optimización': xgb_optimization_info['technique'],
    'Configuración': xgb_optimization_info['comments'],
    'Tiempo de Entrenamiento (s)': training_time_xgb,
    'Uso de Memoria (MB)': memory_used_xgb,
    'Accuracy': accuracy_xgb
})


# --- 9. SVM Lineal ---
# Técnicas de optimización:  LinearSVC 
# Para SVM Lineal, usaremos LinearSVC, que está optimizado para el caso lineal.
svm_linear_optimization_info = {
    'technique': 'LinearSVC',
    'comments': 'Usando LinearSVC para un SVM lineal eficiente. Dual=False recomendado para n_samples > n_features.'
}
svm_linear_model = LinearSVC(random_state=42, dual=False) # dual=False cuando n_samples > n_features
ejecutar_modelo("SVM Lineal", svm_linear_model, X_train, y_train, X_test, y_test, svm_linear_optimization_info)


# --- Resumen de Resultados ---
results_df = pd.DataFrame(results)
print("\n--- Tabla Resumen de Optimización de Modelos ---")
print(results_df)

# Puedes guardar esta tabla en un archivo CSV o Excel
# results_df.to_csv("resultados_optimizacion_ml.csv", index=False)
# results_df.to_excel("resultados_optimizacion_ml.xlsx", index=False)